# 05_evaluation
Compute top-1 and top-5 accuracy on test set.

In [2]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

project_root = os.path.abspath(os.path.join('..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.clip_embedding import load_clip_model, get_image_embedding, load_image
from src.vector_search import load_faiss_index, search_index
from src.dataset import load_metadata
import torch

metadata = load_metadata('../data/metadata.csv')
train_meta, test_meta = train_test_split(metadata, test_size=0.2, stratify=metadata['label'], random_state=42)

# This notebook assumes index already built on full/ train set. It can be adjusted for test with separate index.
index = load_faiss_index('../data/faiss_index.faiss')
model, processor, device = load_clip_model()

def eval_split(test_df, top_k=5):
    top1 = 0
    top5 = 0
    for _, row in test_df.iterrows():
        img = load_image(row['image_path'])
        vec = get_image_embedding(img, processor, model, device=device)
        _, indices = search_index(index, vec, top_k=top_k)
        candidates = test_df.iloc[indices]['label'].values
        if row['label'] == candidates[0]:
            top1 += 1
        if row['label'] in candidates:
            top5 += 1
    n = len(test_df)
    return top1 / n, top5 / n

t1, t5 = eval_split(test_meta, top_k=5)
print(f'Top-1: {t1:.4f} Top-5: {t5:.4f}')

c:\Users\xuann\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: [enforce fail at alloc_cpu.cpp:117] data. DefaultCPUAllocator: not enough memory: you tried to allocate 101187584 bytes.